# AutoGen Example (Using Ollama for LLMs via Colab)

Overall workflow:
1. Check Python version and install system dependencies
2. Install Ollama and start the local inference server
3. Pull the local model (llama3.2:1b)
4. Install AutoGen packages (autogen-agentchat, autogen-ext[openai])
5. RoundRobinGroupChat: a primary agent and a critic agent collaborate to iteratively improve a response until the critic approves
6. Human-In-The-Loop: a human (UserProxyAgent) joins the conversation and can guide or approve the agent's output interactively

In [ ]:
!python3 --version
# Check Python version. AutoGen requires Python 3.10+, so make sure your environment matches.

Python 3.12.12


In [ ]:
!apt-get install -y zstd
# zstd is required to extract the Ollama installation package

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 37 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (415 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
# Install Ollama, which allows us to run LLMs locally instead of using cloud APIs.

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!nohup ollama serve > ollama.log &
# Start the Ollama inference server in the background, with logs redirected to ollama.log.
# nohup ensures the process keeps running even if the Colab session switches tabs.
# AutoGen will communicate with this server at localhost:11434.

nohup: redirecting stderr to stdout


In [ ]:
!ollama pull llama3.2:1b
# Download the llama3.2:1b model locally via Ollama.
# The 1b variant is lightweight and well-suited for Colab's free tier.
# Once pulled, the model runs entirely locally with no external API calls needed.

In [ ]:
!pip install -U "autogen-agentchat"
# Install AutoGen's agent chat package.
# autogen-agentchat provides the core multi-agent conversation framework,
# including agents, teams, and message-passing interfaces.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.3/119.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 12.2 MB/s eta 0:00:00


In [ ]:
!pip install "autogen-ext[openai]"
# Install the OpenAI extension for AutoGen.
# Even when using Ollama locally, this extension is needed because Ollama
# exposes an OpenAI-compatible API endpoint that AutoGen connects through.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 10.3 MB/s eta 0:00:00


## RoundRobinGroupChat

"A team that runs a group chat with participants taking turns in a round-robin fashion."

In [ ]:
# reference: https://microsoft.github.io/autogen/stable//user-guide/agentchat-user-guide/tutorial/teams.html#creating-a-team
import asyncio

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.base import TaskResult
from autogen_agentchat.conditions import ExternalTermination, TextMentionTermination, MaxMessageTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_core import CancellationToken
from autogen_ext.models.openai import OpenAIChatCompletionClient


# Local Ollama: client still requires a key; use placeholder
from autogen_core.models import UserMessage

model_client = OpenAIChatCompletionClient(
    model="llama3.2:1b",
    base_url="http://localhost:11434/v1",
    api_key="placeholder",
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": False,
        "family": "unknown",
    },
)

# Create the primary agent.
primary_agent = AssistantAgent(
    "primary",
    model_client=model_client,
    system_message="You are a helpful AI assistant.",
)

# Create the critic agent.
critic_agent = AssistantAgent(
    "critic",
    model_client=model_client,
    system_message="Provide constructive feedback. Respond with 'APPROVE' to when your feedbacks are addressed.",
)

# Define a termination condition that stops the task if the critic approves.
text_termination = TextMentionTermination("APPROVE")
max_messages_termination = MaxMessageTermination(max_messages=6) # avoid infinite loop for testing, 6 is the max number of messages to be considered for termination
termination = text_termination | max_messages_termination


# Create a team with the primary and critic agents.
team = RoundRobinGroupChat([primary_agent, critic_agent], termination_condition=termination)


async def run_tasks():
    """Run the team"""
    await team.reset()  # Reset the team for a new task.
    await Console(team.run_stream(task="Write a short poem about the fall season."))  # Stream the messages to the console.

# asyncio.run(run_tasks()) for .py scripts
await run_tasks()

/usr/local/lib/python3.12/dist-packages/autogen_ext/models/openai/_openai_client.py:466: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(self._model_info)


---------- TextMessage (user) ----------
Write a short poem about the fall season.
---------- TextMessage (primary) ----------
As autumn leaves begin to sway,
Golden hues and crimsongray.
The wind whispers through the trees,
A soothing melody, wild and free.

The air is crisp, the earth is bare,
Cool breeze that brings delight and care.
The scent of woodsmoke fills the air,
As nature's final dance begins to share.

The stars shine bright in autumn skies,
A canopy of twinkling lights and sighs.
The world is hushed, a peaceful sight,
In all its beauty, a brief, shining light.
---------- TextMessage (critic) ----------
APPROVE


## Human-In-The-Loop

Provide human feedback to the team.

In [ ]:
# reference: https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/tutorial/human-in-the-loop.html

import asyncio
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

# Create the agents.
model_client = OpenAIChatCompletionClient(
    model="llama3.2:1b",
    base_url="http://localhost:11434/v1",
    api_key="placeholder",
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": False,
        "family": "unknown",
    },
)

assistant = AssistantAgent("assistant", model_client=model_client)
user_proxy = UserProxyAgent("user_proxy", input_func=input)  # Use input() to get user input from console.

# Create the termination condition which will end the conversation when the user says "APPROVE".
termination = TextMentionTermination("APPROVE")

# Create the team.
team = RoundRobinGroupChat([assistant, user_proxy], termination_condition=termination)


async def run_tasks():
    """Run the team"""
    await team.reset()  # Reset the team for a new task.
    await Console(team.run_stream(task="Write a short poem about the fall season."))  # Stream the messages to the console.

# asyncio.run(run_tasks()) for .py scripts
await run_tasks()


---------- TextMessage (user) ----------
Write a short poem about the fall season.
---------- TextMessage (assistant) ----------
As autumn leaves begin to fall,
Golden hues and crimson wall.
The air is crisp, the winds do sway,
A season of change each day.

The trees stand tall, their branches bare,
Their final dance before winter's care.
The scent of woodsmoke fills the air,
As nature's canvas starts to share.

The earthy smell of mulled wine,
Fills hearts with warmth and love that clings.
A time for cozy nights and rest,
As fall's departure is truly blessed.
Enter your response: another version please
---------- TextMessage (user_proxy) ----------
another version please
---------- TextMessage (assistant) ----------
Amidst the twilight, leaves once more
Curl golden brown, then fall to shore.
The breeze whispers secrets, soft and low,
Of summer's warmth that begins to go.

The trees stand bare, their limbs outstretched,
A skeletal frame, a winter's sketch.
The ground is carpeted, red a